In [2]:
import duckdb
import pandas as pd

# connect to in-memory DuckDB instance
con = duckdb.connect()

# point to csv folder
CSV_PATH = "../output/csv"

print("DuckDB version:", duckdb.__version__)
print("Connection Established")

DuckDB version: 1.5.3
Connection Established


In [3]:
# preview patients table
patients = con.execute(f"""
    SELECT *
    FROM read_csv_auto('{CSV_PATH}/patients.csv')
    LIMIT 5
""").df()

print("Shape:", patients.shape)
patients.head()

Shape: (5, 28)


,Id,BIRTHDATE,DEATHDATE,SSN,DRIVERS,PASSPORT,PREFIX,FIRST,MIDDLE,LAST,...,CITY,STATE,COUNTY,FIPS,ZIP,LAT,LON,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE,INCOME
0,f87db36b-e075-cbe3-c832-b71993c70147,2023-11-04,NaT,999-45-2699,None,None,None,Rupert654,Damion480,Kris249,...,Springfield,Massachusetts,Hampden County,25013,01106,42.143593,-72.627188,17177.06,927.25,37574
1,c55574e7-4cfe-4b44-94ae-fb8adacf851f,1992-08-14,NaT,999-88-8886,S99980961,X35428415X,Mr.,Garry927,Jarrett354,Schuster709,...,Leominster,Massachusetts,Worcester County,25027,01453,42.561384,-71.862780,105110.03,0.00,64187
2,9a6c40e0-ae30-b4ad-2f31-3019bbb9414a,1976-04-09,NaT,999-54-7112,S99921614,X89582696X,Mr.,Lacy523,Mack300,Grady603,...,Watertown,Massachusetts,Middlesex County,25017,02472,42.391405,-71.154462,92657.79,503.63,136968
3,a4ffd1ae-583f-386e-9cbd-7e28bfedb57a,1987-01-16,NaT,999-87-3305,S99927252,X49292794X,Mr.,Elisha578,Samuel331,Lemke654,...,Somerville,Massachusetts,Middlesex County,25017,02145,42.436346,-71.146902,127306.65,19241.88,130059
4,c7ad321f-5bab-a30a-d912-c85110b6ab72,2009-01-03,NaT,999-11-8101,S99920989,None,None,Maricruz991,Sharri659,Corwin846,...,Charlton,Massachusetts,Worcester County,<NA>,00000,42.118211,-71.965900,22853.80,24302.40,34917


In [5]:
# preview encounters table
encounters = con.execute(f"""
    SELECT *
    FROM read_csv_auto('{CSV_PATH}/encounters.csv')
    LIMIT 5
""").df()

print("Shape:", encounters.shape)
encounters.head

Shape: (5, 15)


<bound method NDFrame.head of                                      Id                     START  \
0  f87db36b-e075-cbe3-e28a-c2409a89a1c6 2023-11-04 01:07:38-07:00   
1  c55574e7-4cfe-4b44-be1b-b822900fe11b 1995-08-05 15:24:50-07:00   
2  c55574e7-4cfe-4b44-3559-eeabbe93ab62 2010-10-08 02:24:50-07:00   
3  c55574e7-4cfe-4b44-8968-d2c95592717d 2011-10-14 02:24:50-07:00   
4  c55574e7-4cfe-4b44-63e3-3f13fa8f4c78 2014-10-17 02:24:50-07:00   

                       STOP                               PATIENT  \
0 2023-11-04 01:22:38-07:00  f87db36b-e075-cbe3-c832-b71993c70147   
1 1995-08-05 15:39:50-07:00  c55574e7-4cfe-4b44-94ae-fb8adacf851f   
2 2010-10-08 03:02:33-07:00  c55574e7-4cfe-4b44-94ae-fb8adacf851f   
3 2011-10-14 03:21:26-07:00  c55574e7-4cfe-4b44-94ae-fb8adacf851f   
4 2014-10-17 03:02:30-07:00  c55574e7-4cfe-4b44-94ae-fb8adacf851f   

                           ORGANIZATION                              PROVIDER  \
0  c5403c4f-14d5-3171-9227-a9b8327f3149  b671c393-6833-3c5f

In [6]:
# encounter class overview
encounter_classes = con.execute(f"""
    SELECT
        ENCOUNTERCLASS,
        COUNT(*) AS encounter_count
    FROM read_csv_auto('{CSV_PATH}/encounters.csv')
    GROUP BY ENCOUNTERCLASS
    ORDER BY encounter_count DESC
""").df()

encounter_classes

,ENCOUNTERCLASS,encounter_count
0,ambulatory,36593
1,wellness,14477
2,outpatient,9243
3,urgentcare,3409
4,emergency,2675
5,inpatient,1212
6,home,281
7,hospice,176
8,snf,166
9,virtual,128


In [7]:
# how many inpatient encounters per patient
inpatient_counts = con.execute(f"""
    SELECT
        PATIENT,
        COUNT(*) AS inpatient_visits
    FROM read_csv_auto('{CSV_PATH}/encounters.csv')
    WHERE ENCOUNTERCLASS = 'inpatient'
    GROUP BY PATIENT
    ORDER BY inpatient_visits DESC
    LIMIT 10
""").df()

print("Total inpatient encounters:", 1212)
inpatient_counts

Total inpatient encounters: 1212


,PATIENT,inpatient_visits
0,2d0a6b86-65d5-8b49-066f-174cfb84c86d,49
1,fa6902fc-f2b5-d24c-4a1d-8b9a0e902088,47
2,5bda4590-86ae-f5e7-ff02-8f2ea2ed9498,45
3,96c6bef3-3e56-ecdd-e416-957a86f08cad,45
4,c89e0415-c1f6-3c2d-aa57-4ba9a1c90d5a,39
5,d7ba6a32-14e6-d114-0dbd-9f358ac01d61,35
6,27b58487-cebb-4f70-446b-86b081d33a69,34
7,04eaeef0-4c04-a404-7dd3-79e4c37edd44,31
8,4aedf460-48fd-5110-1397-c995df5f23b4,31
9,2f3c7505-861d-f8c0-77a1-c2134c0506ce,29


In [8]:
print("Exploration complete.")
print("Total inpatient encounters: 1212")

Exploration complete.
Total inpatient encounters: 1212
